In [0]:
WITH quarter_end AS (
  SELECT
    deployable_account_name,
    fiscal_year_quarter,
    usage_date,
    SUM(dbu_dollars_t28d_sum) AS dbu_dollars_t28d_sum
  FROM main.gtm_gold.account_consumption_daily
  WHERE horizontal_and_vertical_hierarchy_concatenated_emails LIKE '%antonio.carrozzo@databricks.com%'
    AND deployable_account_name IN ('CNH Industrial', 'Iveco', 'EssilorLuxottica (see Luxottica)', 'Poste Italiane', 'Eni')
    AND CAST(SUBSTR(fiscal_year_quarter, 4, 2) AS INT) >= 26
  GROUP BY deployable_account_name, fiscal_year_quarter, usage_date
  QUALIFY ROW_NUMBER() OVER (PARTITION BY deployable_account_name, fiscal_year_quarter ORDER BY usage_date DESC) = 1

)

SELECT
  deployable_account_name,
  fiscal_year_quarter,
  usage_date AS snapshot_date,
  ROUND(dbu_dollars_t28d_sum, 2) AS dbu_dollars_t28d_sum,
  ROUND(LAG(dbu_dollars_t28d_sum) OVER (PARTITION BY deployable_account_name ORDER BY usage_date), 2) AS prev_quarter_t28d_sum,
  ROUND(dbu_dollars_t28d_sum - LAG(dbu_dollars_t28d_sum) OVER (PARTITION BY deployable_account_name ORDER BY usage_date), 2) AS qoq_change,
  ROUND(
    TRY_DIVIDE(
      dbu_dollars_t28d_sum - LAG(dbu_dollars_t28d_sum) OVER (PARTITION BY deployable_account_name ORDER BY usage_date),
      LAG(dbu_dollars_t28d_sum) OVER (PARTITION BY deployable_account_name ORDER BY usage_date)
    ) * 100, 2
  ) AS qoq_change_pct
FROM quarter_end
ORDER BY fiscal_year_quarter desc